# Chapter 5: Foundations of Statistical Inference

**Artificial Intelligence Engineering – Statistics**

This notebook covers the fundamental concepts of statistical inference with hands-on Python examples:

1. Point Estimates and Sampling Variability
2. Sampling Distribution
3. Central Limit Theorem (CLT)
4. Confidence Intervals
5. Hypothesis Testing
6. Decision Errors (Type 1 / Type 2)

---

In [ ]:
# Required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

np.random.seed(42)
print("Libraries loaded successfully.")

---
## 1. Point Estimates and Sampling Variability

**Theory:**
- We cannot directly measure population parameters (e.g. mean, proportion).
- We use statistics computed from a sample as **point estimates**.
- **Bias:** Systematic deviation of the estimated value.
- **Sampling error:** Variation of estimates from sample to sample.

### 1.1 Example: Average Height in the US

If 1000 adults were sampled from each state, the average heights would be very close to each other but not identical.

In [ ]:
# True population: mean 175 cm, std 10 cm
POP_MEAN   = 175
POP_STD    = 10
SAMPLE_SIZE = 1000
N_STATES   = 50

# Simulate sample mean for each state
state_means = [
    np.mean(np.random.normal(POP_MEAN, POP_STD, SAMPLE_SIZE))
    for _ in range(N_STATES)
]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(1, N_STATES + 1), state_means,
       color='steelblue', edgecolor='white', alpha=0.85)
ax.axhline(POP_MEAN, color='red', linestyle='--',
           linewidth=2, label=f'True population mean = {POP_MEAN} cm')
ax.set_xlabel('State')
ax.set_ylabel('Sample Mean (cm)')
ax.set_title('Height Means Obtained from 50 States (n=1000)')
ax.set_ylim(172, 178)
ax.legend()
plt.tight_layout()
plt.show()

print(f"Range of sample means: [{min(state_means):.2f}, {max(state_means):.2f}]")
print(f"True population mean: {POP_MEAN} cm")
print("→ All estimates are very close to each other but not exactly the same!")

---
## 2. Sampling Distribution

**Theory:**
The distribution formed by all the $\hat{p}$ values obtained when the same procedure is repeated many times is called the **sampling distribution**.

> In the real world we can never observe the sampling distribution; however, it is very useful to think of every point estimate as coming from this hypothetical distribution.

### 2.1 Solar Energy Example (Simulation from Slides)

True population proportion **p = 0.88** (Americans who support solar energy).

In [ ]:
# Population parameter
p_true    = 0.88
n         = 1000        # sample size
n_repeats = 10000       # number of repeated samples

# Draw a sample each time and compute p-hat
p_hat_list = np.random.binomial(n, p_true, n_repeats) / n

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(p_hat_list, bins=30, color='steelblue',
        edgecolor='white', alpha=0.85, density=False)
ax.axvline(p_true, color='red', linestyle='--',
           linewidth=2.5, label=f'True p = {p_true}')
ax.axvline(np.mean(p_hat_list), color='orange', linestyle='-.',
           linewidth=2, label=f'Sampling mean = {np.mean(p_hat_list):.3f}')
ax.set_xlabel('Simulated sample proportion ($\hat{p}$)')
ax.set_ylabel('Frequency')
ax.set_title('Sampling Distribution (p=0.88, n=1000, 10,000 repetitions)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Mean of sampling distribution   : {np.mean(p_hat_list):.4f}")
print(f"Std dev of sampling distribution: {np.std(p_hat_list):.4f}")
print(f"Theoretical standard error      : {np.sqrt(p_true*(1-p_true)/n):.4f}")

---
## 3. Central Limit Theorem (CLT)

**Theory:**
$$\hat{p} \sim N\left(p,\ SE = \sqrt{\frac{p(1-p)}{n}}\right)$$

**CLT Conditions:**
1. **Independence:** Observations must be independent (random sampling + n < 10% of population)
2. **Sample size:** $np \geq 10$ **and** $n(1-p) \geq 10$

### 3.1 Effect of n and p on the Shape of the Distribution

In [ ]:
p_values = [0.1, 0.2, 0.5, 0.8, 0.9]
n_values = [10, 25, 50, 100, 250]
n_sim    = 3000

fig, axes = plt.subplots(len(p_values), len(n_values),
                         figsize=(16, 12), sharex=True)

for i, p in enumerate(p_values):
    for j, n_val in enumerate(n_values):
        simulation = np.random.binomial(n_val, p, n_sim) / n_val
        axes[i, j].hist(simulation, bins=20, color='steelblue',
                        edgecolor='none', alpha=0.8, density=True)
        axes[i, j].set_xlim(0, 1)
        axes[i, j].set_yticks([])
        axes[i, j].tick_params(labelsize=7)

        # Condition check
        condition_met = (n_val * p >= 10) and (n_val * (1 - p) >= 10)
        border_color  = 'green' if condition_met else 'red'
        axes[i, j].set_title(f'n={n_val}', fontsize=8, color='black')
        for sp in axes[i, j].spines.values():
            sp.set_edgecolor(border_color)
            sp.set_linewidth(2)

        if j == 0:
            axes[i, j].set_ylabel(f'p={p}', fontsize=9)

# Legend
green_patch = mpatches.Patch(color='green', label='CLT condition met (np≥10 and n(1-p)≥10)')
red_patch   = mpatches.Patch(color='red',   label='CLT condition NOT met')
fig.legend(handles=[green_patch, red_patch], loc='lower center',
           ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.01))

fig.suptitle('Shape of the Sampling Distribution When np and n(1-p) < 10',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── CLT Condition Check – Helper Function ────────────────────────────────────
def clt_condition_check(n, p=None, p_hat=None, population_size=None):
    """
    Checks CLT conditions.
    If p is unknown, p_hat is used.
    """
    p_use = p if p is not None else p_hat
    if p_use is None:
        raise ValueError("Either p or p_hat must be provided.")

    print("=" * 50)
    print("CLT Condition Check")
    print("=" * 50)

    # Condition 1: Independence / 10% rule
    if population_size:
        ratio = n / population_size
        c1    = ratio < 0.10
        print(f"1) 10% rule: n/N = {ratio:.4f} → {'✓ MET' if c1 else '✗ NOT MET'}")
    else:
        print("1) 10% rule: Population size unknown, random sampling assumed.")
        c1 = True

    # Condition 2: Success-failure condition
    np_val  = n * p_use
    n1p_val = n * (1 - p_use)
    c2      = (np_val >= 10) and (n1p_val >= 10)
    print(f"2) Success-failure condition:")
    print(f"   np     = {n} × {p_use:.3f} = {np_val:.1f}  → {'✓' if np_val >= 10 else '✗ (< 10!)'}")
    print(f"   n(1-p) = {n} × {1-p_use:.3f} = {n1p_val:.1f}  → {'✓' if n1p_val >= 10 else '✗ (< 10!)'}")

    result = c1 and c2
    print("-" * 50)
    print(f"RESULT: CLT {'APPLICABLE ✓' if result else 'NOT APPLICABLE ✗'}")
    print("=" * 50)
    return result

# Solar energy example
clt_condition_check(n=1000, p=0.88, population_size=250_000_000)

In [ ]:
# p=0.05, n=50 → condition NOT MET
print("Example from slides: p=0.05, n=50")
clt_condition_check(n=50, p=0.05)

# Visualization
p_low, n_small = 0.05, 50
sim_low = np.random.binomial(n_small, p_low, 5000) / n_small

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(sim_low, bins=15, color='tomato', edgecolor='white', alpha=0.85)
ax.set_xlabel('Sample proportion ($\hat{p}$)')
ax.set_ylabel('Frequency')
ax.set_title(f'p={p_low}, n={n_small} → Skewed distribution (condition not met!)')
ax.text(0.10, 200, f'np = {n_small*p_low:.1f} < 10\n→ Normal approximation invalid!',
        fontsize=11, color='darkred',
        bbox=dict(boxstyle='round', facecolor='mistyrose', alpha=0.8))
plt.tight_layout()
plt.show()

---
## 4. Confidence Intervals

**Theory:**
$$\hat{p} \pm z^* \times SE, \qquad SE = \sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$$

| Confidence Level | z* |
|:---:|:---:|
| 90% | 1.645 |
| 95% | 1.960 |
| 98% | 2.326 |
| 99% | 2.576 |

### 4.1 Facebook Interest Categories Example (Calculation from Slides)

In [ ]:
def confidence_interval(p_hat, n, confidence=0.95):
    """
    Computes a confidence interval for a proportion.
    """
    alpha  = 1 - confidence
    z_star = stats.norm.ppf(1 - alpha / 2)
    se     = np.sqrt(p_hat * (1 - p_hat) / n)
    margin = z_star * se
    lower  = p_hat - margin
    upper  = p_hat + margin

    print(f"{'='*55}")
    print(f"Confidence Interval Calculation ({confidence*100:.0f}%)")
    print(f"{'='*55}")
    print(f"  p̂ (sample proportion)  : {p_hat}")
    print(f"  n (sample size)        : {n}")
    print(f"  z* (critical value)    : {z_star:.4f}")
    print(f"  SE                     : √({p_hat}×{1-p_hat}/{n}) = {se:.4f}")
    print(f"  Margin of error        : {z_star:.4f} × {se:.4f} = {margin:.4f} ≈ {margin:.2f}")
    print(f"-{'-'*54}")
    print(f"  Confidence interval    : ({lower:.4f}, {upper:.4f})"
          f"  ≈  ({lower*100:.1f}%, {upper*100:.1f}%)")
    print(f"{'='*55}")
    return lower, upper, z_star, se

# Example from slides
lower, upper, _, _ = confidence_interval(p_hat=0.67, n=850, confidence=0.95)

In [ ]:
# Comparison of different confidence levels
p_hat, n         = 0.67, 850
conf_levels      = [0.90, 0.95, 0.98, 0.99]
colors           = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']

fig, ax = plt.subplots(figsize=(11, 5))

for i, (conf, color) in enumerate(zip(conf_levels, colors)):
    z     = stats.norm.ppf(1 - (1 - conf) / 2)
    se    = np.sqrt(p_hat * (1 - p_hat) / n)
    lower = p_hat - z * se
    upper = p_hat + z * se

    ax.plot([lower, upper], [i, i], color=color, linewidth=5, solid_capstyle='round')
    ax.plot(p_hat, i, 'o', color='black', markersize=7, zorder=5)
    ax.text(upper + 0.003, i, f'{conf*100:.0f}%  [{lower:.3f}, {upper:.3f}]',
            va='center', fontsize=10, color=color)

ax.axvline(p_hat, color='black', linestyle='--', alpha=0.4, label=f'p̂ = {p_hat}')
ax.set_yticks(range(len(conf_levels)))
ax.set_yticklabels([f'{int(g*100)}%' for g in conf_levels])
ax.set_xlabel('Proportion')
ax.set_title('Confidence Intervals at Different Confidence Levels (p̂=0.67, n=850)')
ax.set_xlim(0.60, 0.78)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()
print("As the confidence level increases the interval WIDENS → provides less precise information!")

### 4.2 What Does '95% Confidence' Mean?

When many samples are taken, approximately **95%** of the constructed intervals will contain the true population parameter.

In [ ]:
# Simulation of the meaning of 95% confidence
p_true      = 0.67
n           = 850
n_samples   = 100   # number of different samples
confidence  = 0.95
z_star      = stats.norm.ppf(1 - (1 - confidence) / 2)

coverage_count = 0
fig, ax = plt.subplots(figsize=(12, 8))

for i in range(n_samples):
    phat   = np.random.binomial(n, p_true) / n
    se     = np.sqrt(phat * (1 - phat) / n)
    lower  = phat - z_star * se
    upper  = phat + z_star * se
    covers = lower <= p_true <= upper

    if covers:
        coverage_count += 1

    color = "#4CAF50" if covers else "#F44336"
    ax.plot([lower, upper], [i, i], color=color, linewidth=1.5,
            alpha=0.75, solid_capstyle="round")

ax.axvline(p_true, color="black", linewidth=2.5,
           linestyle="--", label=f"True p = {p_true}")

green_p = mpatches.Patch(color="#4CAF50", label=f"Contains ({coverage_count}/{n_samples})")
red_p   = mpatches.Patch(color="#F44336", label=f"Does not contain ({n_samples - coverage_count}/{n_samples})")
ax.legend(handles=[green_p, red_p,
                   mpatches.Patch(color="black", label=f"True p={p_true}")],
          fontsize=10)

ax.set_xlabel("Proportion")
ax.set_ylabel("Sample")
ax.set_title(f"95% Confidence Interval: {coverage_count} out of 100 samples contain the true p")
ax.set_yticks([])
plt.tight_layout()
plt.show()

print("Coverage rate:", coverage_count, "/", n_samples, "=", coverage_count, "%")
print("Theoretical expectation: 95%")

---
## 5. Hypothesis Testing

**Framework:**
- $H_0$: Null hypothesis (status quo)
- $H_A$: Alternative hypothesis (research question)

$$Z = \frac{\hat{p} - p_0}{SE_0}, \qquad SE_0 = \sqrt{\frac{p_0(1-p_0)}{n}}$$

### 5.1 Facebook Interest Categories Hypothesis Test (Example from Slides)

$H_0: p = 0.50$ vs $H_A: p \neq 0.50$, $\hat{p} = 0.41$, $n = 850$

In [ ]:
def hypothesis_test_proportion(p_hat, n, p_0, direction='two_sided', alpha=0.05):
    """
    Z-test for a proportion.
    direction: 'two_sided', 'left', 'right'
    """
    se_0 = np.sqrt(p_0 * (1 - p_0) / n)
    z    = (p_hat - p_0) / se_0

    if direction == 'two_sided':
        p_value = 2 * stats.norm.sf(abs(z))
        h_a     = f'p ≠ {p_0}'
    elif direction == 'left':
        p_value = stats.norm.cdf(z)
        h_a     = f'p < {p_0}'
    else:
        p_value = stats.norm.sf(z)
        h_a     = f'p > {p_0}'

    print(f"{'='*55}")
    print("Hypothesis Test")
    print(f"{'='*55}")
    print(f"  H₀ : p = {p_0}")
    print(f"  Hₐ : {h_a}")
    print(f"  p̂  = {p_hat},  n = {n},  α = {alpha}")
    print("-" * 55)
    print(f"  SE₀ = √({p_0}×{1-p_0}/{n}) = {se_0:.4f}")
    print(f"  Z   = ({p_hat} - {p_0}) / {se_0:.4f} = {z:.4f}")
    print(f"  p-value = {p_value:.6f}")
    print("-" * 55)
    if p_value < alpha:
        print(f"  DECISION: p-value ({p_value:.4f}) < α ({alpha})")
        print(f"  → REJECT H₀. There is sufficient evidence in favour of Hₐ.")
    else:
        print(f"  DECISION: p-value ({p_value:.4f}) ≥ α ({alpha})")
        print(f"  → FAIL TO REJECT H₀. Insufficient evidence for Hₐ.")
    print(f"{'='*55}")
    return z, p_value

# Example from slides
z_stat, p_value = hypothesis_test_proportion(p_hat=0.41, n=850, p_0=0.50, direction='two_sided')

In [ ]:
# p-value visualization
fig, ax = plt.subplots(figsize=(11, 5))

x = np.linspace(-7, 7, 1000)
y = stats.norm.pdf(x)

ax.plot(x, y, 'k-', linewidth=2)

# Two tails: |Z| > 5.26
z_crit = abs(z_stat)
x_left  = x[x <= -z_crit]
x_right = x[x >= z_crit]
ax.fill_between(x_left,  stats.norm.pdf(x_left),  color='tomato', alpha=0.7,
                label=f'p-value < 0.0001')
ax.fill_between(x_right, stats.norm.pdf(x_right), color='tomato', alpha=0.7)

# α = 0.05 critical values
z_alpha = stats.norm.ppf(0.975)
ax.axvline(-z_alpha, color='steelblue', linestyle='--', linewidth=1.5,
           label=f'α=0.05 critical value: ±{z_alpha:.2f}')
ax.axvline( z_alpha, color='steelblue', linestyle='--', linewidth=1.5)

ax.axvline(z_stat, color='darkred', linewidth=2.5,
           label=f'Test statistic Z = {z_stat:.2f}')

ax.set_xlabel('Z')
ax.set_ylabel('Density')
ax.set_title('Hypothesis Test: p-value Visualization\n'
             'H₀: p=0.50  vs  Hₐ: p≠0.50  (p̂=0.41, n=850)')
ax.legend(fontsize=10)
ax.set_xlim(-7, 7)
plt.tight_layout()
plt.show()

### 5.2 One-Sided vs Two-Sided Test

**Facebook confidence interval example from slides:**  
Confidence interval (64%, 70%) → because $p_0 = 0.50$ falls **outside** the interval, $H_0$ is rejected.

In [ ]:
def ci_based_test(p_hat, n, p_0, confidence=0.95):
    """Hypothesis test using a confidence interval."""
    lower, upper, _, _ = confidence_interval(p_hat, n, confidence)
    print(f"\nH₀: p = {p_0}")
    print(f"Is the null value ({p_0}) inside the interval? ", end="")
    if lower <= p_0 <= upper:
        print(f"YES → Fail to reject H₀.")
    else:
        print(f"NO → REJECT H₀!")

ci_based_test(p_hat=0.67, n=850, p_0=0.50, confidence=0.95)

# Visualization
fig, ax = plt.subplots(figsize=(9, 3))
z_star  = stats.norm.ppf(0.975)
se      = np.sqrt(0.67 * 0.33 / 850)
lower2  = 0.67 - z_star * se
upper2  = 0.67 + z_star * se

ax.plot([lower2, upper2], [0, 0], color='steelblue', linewidth=10,
        solid_capstyle='round', label=f'95% CI: [{lower2:.3f}, {upper2:.3f}]')
ax.plot(0.67, 0, 'o', color='black', markersize=10, zorder=5, label='p̂ = 0.67')
ax.axvline(0.50, color='red', linewidth=2.5, linestyle='--',
           label='H₀: p₀ = 0.50')
ax.set_yticks([])
ax.set_xlabel('Proportion')
ax.set_title('Hypothesis Test via Confidence Interval\np₀=0.50 is outside the interval → H₀ rejected!')
ax.legend(fontsize=10)
ax.set_xlim(0.44, 0.78)
plt.tight_layout()
plt.show()

---
## 6. Decision Errors

| | Fail to reject H₀ | Reject H₀ |
|:---|:---:|:---:|
| **H₀ true** | ✓ Correct decision | ✗ Type 1 Error (α) |
| **Hₐ true** | ✗ Type 2 Error (β) | ✓ Correct decision |

- **Type 1 Error:** Rejecting H₀ when it is true (false alarm)
- **Type 2 Error:** Failing to reject H₀ when Hₐ is true (missed detection)

$$P(\text{Type 1 Error} \mid H_0 \text{ true}) = \alpha$$

In [ ]:
# Decision errors visualization
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x    = np.linspace(-5, 8, 1000)
mu_0, mu_a = 0, 3   # H₀ and Hₐ distribution means
sigma  = 1
z_crit = stats.norm.ppf(0.95)  # one-sided α=0.05

y0 = stats.norm.pdf(x, mu_0, sigma)
ya = stats.norm.pdf(x, mu_a, sigma)

# Left panel: Type 1 Error
ax = axes[0]
ax.plot(x, y0, 'steelblue', linewidth=2.5, label='H₀ distribution')
x_type1 = x[x >= z_crit]
ax.fill_between(x_type1, stats.norm.pdf(x_type1, mu_0, sigma),
                color='tomato', alpha=0.7, label=f'Type 1 Error (α={0.05})')
ax.axvline(z_crit, color='black', linestyle='--', linewidth=1.5)
ax.set_title('Type 1 Error\nRejecting H₀ when H₀ is true')
ax.set_xlabel('Test statistic')
ax.legend(fontsize=9)
ax.set_yticks([])
ax.text(z_crit + 0.1, 0.15, 'Rejection\nregion', fontsize=9)

# Right panel: Type 2 Error
ax = axes[1]
ax.plot(x, y0, 'steelblue', linewidth=2.5, label='H₀ distribution', alpha=0.6)
ax.plot(x, ya, 'darkorange', linewidth=2.5, label='Hₐ distribution')
x_type2 = x[x <= z_crit]
ax.fill_between(x_type2, stats.norm.pdf(x_type2, mu_a, sigma),
                color='mediumpurple', alpha=0.6,
                label=f'Type 2 Error (β = {stats.norm.cdf(z_crit, mu_a, sigma):.3f})')
ax.axvline(z_crit, color='black', linestyle='--', linewidth=1.5)
ax.set_title('Type 2 Error\nFailing to reject H₀ when Hₐ is true')
ax.set_xlabel('Test statistic')
ax.legend(fontsize=9)
ax.set_yticks([])

plt.suptitle('Decision Errors: Type 1 and Type 2', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Trade-off between α and β: as α decreases, β increases
alpha_values   = np.linspace(0.001, 0.20, 200)
mu_0_val, mu_a_val, sigma_val = 0, 2, 1

beta_values = []
for a in alpha_values:
    z_c  = stats.norm.ppf(1 - a, mu_0_val, sigma_val)
    beta = stats.norm.cdf(z_c, mu_a_val, sigma_val)
    beta_values.append(beta)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(alpha_values, beta_values, 'tomato', linewidth=2.5, label='Type 2 Error (β)')
ax.plot(alpha_values, 1 - np.array(beta_values), 'steelblue',
        linewidth=2.5, label='Power = 1 - β')
ax.axvline(0.05, color='gray', linestyle='--', label='α = 0.05 (common threshold)')
ax.set_xlabel('Significance Level (α = Type 1 Error Rate)')
ax.set_ylabel('Probability')
ax.set_title('As α Decreases, the Type 2 Error Probability (β) Increases')
ax.legend(fontsize=10)
ax.set_xlim(0, 0.20)
plt.tight_layout()
plt.show()
print("Reducing α decreases Type 1 error but increases Type 2 error. A balance must be struck!")

---
## 7. Comprehensive Application: Step-by-Step Hypothesis Test

Starting from the gender discrimination experiment in the slides, let us summarise all steps in a single function.

In [ ]:
def full_inference(p_hat, n, p_0=0.50, confidence=0.95, direction='two_sided', alpha=0.05):
    """
    Full statistical inference for a proportion:
    1) CLT condition check
    2) Confidence interval
    3) Hypothesis test
    """
    print("\n" + "#" * 60)
    print("  STEP 1: CLT CONDITION CHECK")
    print("#" * 60)
    clt_condition_check(n=n, p_hat=p_hat)

    print("\n" + "#" * 60)
    print(f"  STEP 2: {int(confidence*100)}% CONFIDENCE INTERVAL")
    print("#" * 60)
    confidence_interval(p_hat=p_hat, n=n, confidence=confidence)

    print("\n" + "#" * 60)
    print("  STEP 3: HYPOTHESIS TEST")
    print("#" * 60)
    hypothesis_test_proportion(p_hat=p_hat, n=n, p_0=p_0, direction=direction, alpha=alpha)

# Facebook interest categories: 41% comfortable, n=850
# H₀: p=0.50  Hₐ: p≠0.50
full_inference(p_hat=0.41, n=850, p_0=0.50, direction='two_sided')

---
## 8. Summary and Concept Map

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║        CHAPTER 5 – STATISTICAL INFERENCE SUMMARY                ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  POINT ESTIMATE                                                  ║
║    p̂ = sample proportion → best estimate for p (pop. prop.)     ║
║                                                                  ║
║  SAMPLING DISTRIBUTION (CLT)                                     ║
║    p̂ ~ N(p, SE)    SE = √(p(1-p)/n)                             ║
║    Conditions: independence + np ≥ 10 + n(1-p) ≥ 10             ║
║                                                                  ║
║  CONFIDENCE INTERVAL                                             ║
║    p̂ ± z* × SE                                                  ║
║    90%: z*=1.645  |  95%: z*=1.960                              ║
║    98%: z*=2.326  |  99%: z*=2.576                              ║
║                                                                  ║
║  HYPOTHESIS TEST                                                 ║
║    Z = (p̂ - p₀) / SE₀                                           ║
║    p-value < α → Reject H₀                                       ║
║    p-value ≥ α → Fail to reject H₀                               ║
║                                                                  ║
║  DECISION ERRORS                                                 ║
║    Type 1 (α): Rejecting H₀ when it is true                     ║
║    Type 2 (β): Failing to reject H₀ when Hₐ is true            ║
║    α ↓ → β ↑  (inverse trade-off)                               ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
""")

---
## 9. Exercises

### Exercise 1
A survey shows that 62% of a sample of 500 people are satisfied with a product.
- (a) Check the CLT conditions.
- (b) Compute the 95% confidence interval.
- (c) Test whether product satisfaction is greater than 50% (α=0.05).

### Exercise 2
For a true population proportion p=0.3, plot the sampling distributions for n=20, 50, 100, 500 and visually interpret the CLT condition.

### Exercise 3
In the following scenarios, which type of error is more dangerous? Why?
- (a) Cancer screening test
- (b) Spam e-mail filter
- (c) Aircraft component quality control

In [ ]:
# Exercise 1 – Solution area
# (a) CLT conditions
clt_condition_check(n=500, p_hat=0.62)

# (b) Confidence interval
confidence_interval(p_hat=0.62, n=500, confidence=0.95)

# (c) Hypothesis test: H₀: p=0.50, Hₐ: p>0.50 (one-sided)
hypothesis_test_proportion(p_hat=0.62, n=500, p_0=0.50, direction='right', alpha=0.05)